## Creating MCP- Langchain agent for accessing MongoDB 

In [ ]:
import os 
from dotenv import load_dotenv 
load_dotenv() 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning)

from langchain.chat_models import init_chat_model

gemma = init_chat_model(model="gemma4:latest", model_provider="ollama")

#llm = init_chat_model(model="qwen/qwen3-32b", model_provider="Groq")
llm_primary = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")
llm_fallback_1 = init_chat_model(model="gpt-5.4-nano", model_provider="OpenAI")
llm_fallback_2 = init_chat_model(model="gpt-5.4-mini", model_provider="OpenAI")


In [ ]:
MONGODB_URI = os.getenv("MONGODB_URI")

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

## Connect your client with the MongoDB-MCP-server 

In [ ]:
client = MultiServerMCPClient({
"mongodb":{
    "transport":"stdio",
    "command":"npx",
    "args":[
        "-y",
        "mongodb-mcp-server@latest",
        "--loggers",
        "stderr"
    ],
    "env":{
        "MDB_MCP_CONNECTION_STRING":MONGODB_URI
    }
},
})

In [ ]:
tools_mongoDB = await client.get_tools()

In [ ]:
for tool in tools_mongoDB: 
    print(tool.name)

## Create Langchain agent with mcp_tools

In [ ]:
from langchain.agents import create_agent

prompt="""You are a helpful mongodb assistant. 
use tools_mongoDB for connecting and acceesing mongodb database and answer based on user query.
"""

agent= create_agent(
    model=  llm_fallback_1, #gemma,
    tools=tools_mongoDB,
    system_prompt=prompt
)

## Test the agent

In [ ]:
user_query= """How many documents are there in the collection?
 database: University, collection: students
 """

In [ ]:
user_query= """Show me the document information with field information:
name:James Cercone
for database: University, collection: students
"""

In [ ]:
user_query= """Modify the following record:
current name:James Cercone modify to name: James Chase 
for database: University, collection: students
"""

In [ ]:
user_query= """Add the following new document:
name:Ethan Nawaz 
dept_name:Computer Science
gpa:3.99
credit_hours:112 
for database: University, collection: students
"""

In [ ]:
user_query= """Delete the following document:
name:Ethan Nawaz  
for database: University, collection: students
"""

In [ ]:
from langchain.messages import SystemMessage,HumanMessage

try:
    result = await agent.ainvoke({
        "messages":[
            SystemMessage(content="You a helpful assistant."),
            HumanMessage(content=user_query)
        ]
    })
except Exception as e:
    print(f"Error happened during invoke. {e}")

In [ ]:
print(result["messages"][-1].content)

## gmail access

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

google_access_token = os.getenv("GOOGLE_ACCESS_TOKEN")

print("Token exists:", google_access_token is not None)

if google_access_token:
    print("Token starts with:", google_access_token[:15])
    print("Token length:", len(google_access_token))

In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.compose",
]

flow = InstalledAppFlow.from_client_secrets_file(
    "../../google_client_secret.json",
    SCOPES
)

credentials = flow.run_local_server(port=0)

print("Access token:", credentials.token)
print("Refresh token:", credentials.refresh_token)

In [ ]:
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

google_access_token = os.getenv("GOOGLE_ACCESS_TOKEN")

client_gmail = MultiServerMCPClient(
    {
        "gmail": {
            "transport": "http",
            "url": "https://gmailmcp.googleapis.com/mcp/v1",
            "headers": {
                "Authorization": f"Bearer {google_access_token}"
            }
        }
    }
)



In [ ]:
tools = await client_gmail.get_tools()

In [ ]:
for tool in tools:
    print(tool.name)